# Chapter 22: Computer Graphics in Games

Source orientation: printed pages 623-644; physical PDF pages 640-661. I used the source span for topic order and vocabulary, then built this notebook as an original, standalone computational lesson. No textbook figures, crops, screenshots, or exercise text are used.

## Chapter goal

A game renderer is a negotiation between appearance, responsiveness, platform limits, tool-chain limits, and the production team. The goal of this chapter is to make that negotiation inspectable. We will model frame budgets, storage budgets, game-type choices, culling, level of detail, texture and normal-map costs, asset-pipeline dependencies, and profiling decisions with small synthetic experiments.

The central lesson is not that one graphics technique is always best. It is that every technique spends one or more scarce resources: CPU time, GPU time, bandwidth, memory, artist time, latency, or schedule. A useful graphics decision names the resource it spends, names the resource it saves, and checks whether the bottleneck actually moved.

## Translation guide: computational language

| Chapter idea | Computational translation | What we will check |
| --- | --- | --- |
| Platform | A hardware/API/input/deployment profile with budgets and variance | frame milliseconds, memory capacity, bandwidth, abstraction, and variance are explicit fields |
| Frame budget | `1000 / fps` milliseconds divided across CPU, GPU, bandwidth, and synchronization | 60 fps has half the frame time of 30 fps; the slowest resource controls the frame |
| Command buffer | A queue between CPU submission and GPU execution | buffering prevents starvation but increases input-to-display latency |
| Heterogeneous GPU resources | Separate synthetic timings for vertex, pixel, texture, bandwidth, and post effects | reducing non-bottleneck work does not improve frame time |
| Culling | Early rejection of objects outside the view or hidden by scene structure | no rejected object should pass the visibility predicate |
| LOD | A distance or screen-coverage rule selecting cheaper representations | selected triangle counts decrease as screen coverage decreases |
| Texture and normal maps | Arrays with byte cost, mip chains, and encoded normals | mip cost is near the geometric 4/3 rule; decoded normals are unit length |
| Asset pipeline | A dependency graph from concept through modeling, texturing, shading, lighting, animation, QA, and release | the graph is acyclic and exposes critical dependencies |
| Profiling | Synthetic measurements compared to target budgets | recommended quality settings fit the target frame and memory budgets |

## Visual storyboard

1. `platform-resource-envelope.png`: compare fixed consoles, variable PCs, browser/VM games, and handheld/mobile targets as resource envelopes rather than as names.
2. `cpu-gpu-command-buffer-latency.png`: show why command buffering reduces GPU starvation but adds at least one frame of latency.
3. `game-type-rendering-choice-map.html`: interactive Plotly map connecting game type to frame rate, world span, visible actors, precomputation, streaming pressure, and asset variety.
4. `culling-lod-visibility-budget.png`: synthetic scene experiment for frustum culling, occlusion culling, LOD selection, triangle count, and pixel work.
5. `texture-normal-map-memory-budget.png` and `synthetic-baked-normal-map.png`: mip-chain and normal-map budget checks with a generated normal field.
6. `asset-pipeline-production-dag.png`: production and art-asset dependency graph showing where graphics decisions become tool and schedule decisions.
7. `profiling-optimization-lab.png`: quality-knob search that chooses settings from synthetic performance measurements.

Each artifact is saved under `artifacts/chapter-22/` and displayed near the explanation that uses it. The checks at the end assert that artifacts exist, are nonempty, and preserve the chapter-specific invariants.


In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px

HERE = Path.cwd().resolve()
for candidate in [HERE, *HERE.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not locate the Fundamentals of Computer Graphics book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, artifact_path, display_artifact, save_image, save_json, save_matplotlib, save_plotly_html, save_table_csv
from utils.plotting import PALETTE, image_stats, style_axis

TOPIC = "chapter-22"
SOURCE_SPAN = {
    "title": "Computer Graphics in Games",
    "chapter": 22,
    "printed_pages": "623-644",
    "pdf_pages": "640-661",
    "sections": ["22.1 Platforms", "22.2 Limited Resources", "22.3 Optimization Techniques", "22.4 Game Types", "22.5 The Game Production Process"],
}

for kind in ["figures", "html", "checks", "tables", "data"]:
    artifact_path(TOPIC, kind, ".keep")

storyboard = {
    "chapter_goal": "Model game graphics decisions as budgeted, profiled, platform-specific tradeoffs.",
    "source_span_read": SOURCE_SPAN,
    "visuals": [
        {"artifact": "figures/platform-resource-envelope.png", "concept": "platform and resource constraints", "check": "positive frame, memory, and bandwidth budgets"},
        {"artifact": "figures/cpu-gpu-command-buffer-latency.png", "concept": "processing time and command buffering", "check": "frame time is controlled by max resource time"},
        {"artifact": "html/game-type-rendering-choice-map.html", "concept": "game type rendering choices", "check": "latency-sensitive types use tighter frame budgets"},
        {"artifact": "figures/culling-lod-visibility-budget.png", "concept": "culling and LOD", "check": "LOD triangle count is monotone with distance bands"},
        {"artifact": "figures/texture-normal-map-memory-budget.png", "concept": "texture and normal-map budgets", "check": "mip chain ratio and normal lengths are bounded"},
        {"artifact": "figures/asset-pipeline-production-dag.png", "concept": "asset pipeline and production workflow", "check": "dependency graph is acyclic"},
        {"artifact": "figures/profiling-optimization-lab.png", "concept": "profiling and quality settings", "check": "chosen settings satisfy target budgets"},
    ],
}
storyboard_path = save_json(storyboard, TOPIC, "visual-storyboard.json")
source_span_path = save_json(SOURCE_SPAN, TOPIC, "source-span.json")
print(f"Storyboard: {storyboard_path.relative_to(BOOK_ROOT)}")


## Platforms as resource envelopes

The chapter treats a platform as more than a GPU name. A platform combines hardware, operating system, API, input devices, distribution rules, and performance expectations. That is why a fixed console, a variable Windows PC, a browser virtual machine, and a handheld target lead to different graphics choices even if they can all draw textured triangles.

The figure below turns platform descriptions into a small budget table. The numbers are synthetic and normalized, so they are not product specifications. Their purpose is to make the shape of the decision visible. A fixed console has low variance and predictable timing, but a strict memory and certification environment. A high-end PC has more capacity, but the developer must support a wide range of machines. A browser or VM target gains deployment reach and API stability at the cost of lower performance access. A handheld target may have capable hardware, but thermal and bandwidth pressure make sustained rendering budgets tight.

Inspect the heatmap row by row. The useful question is not "which row is best?" but "which constraint should drive the renderer and content pipeline?"


In [ ]:
platform_profiles = pd.DataFrame([
    {"platform": "Fixed console", "target_fps": 30, "time_budget_ms": 1000 / 30, "gpu_capacity": 7.0, "memory_gb": 16, "bandwidth_gbps": 180, "api_abstraction": 3, "platform_variance": 1, "certification_pressure": 9},
    {"platform": "PC low spec", "target_fps": 30, "time_budget_ms": 1000 / 30, "gpu_capacity": 3.0, "memory_gb": 8, "bandwidth_gbps": 80, "api_abstraction": 7, "platform_variance": 8, "certification_pressure": 2},
    {"platform": "PC high spec", "target_fps": 60, "time_budget_ms": 1000 / 60, "gpu_capacity": 10.0, "memory_gb": 32, "bandwidth_gbps": 450, "api_abstraction": 7, "platform_variance": 9, "certification_pressure": 2},
    {"platform": "Browser or VM", "target_fps": 30, "time_budget_ms": 1000 / 30, "gpu_capacity": 2.0, "memory_gb": 4, "bandwidth_gbps": 30, "api_abstraction": 10, "platform_variance": 6, "certification_pressure": 4},
    {"platform": "Handheld mobile", "target_fps": 30, "time_budget_ms": 1000 / 30, "gpu_capacity": 4.0, "memory_gb": 6, "bandwidth_gbps": 60, "api_abstraction": 6, "platform_variance": 5, "certification_pressure": 8},
])
platform_profiles["frame_time_tightness"] = 16.667 / platform_profiles["time_budget_ms"]
platform_profiles["streaming_pressure"] = 1 / np.sqrt(platform_profiles["memory_gb"] * platform_profiles["bandwidth_gbps"] / 100)

resource_cols = ["frame_time_tightness", "gpu_capacity", "memory_gb", "bandwidth_gbps", "api_abstraction", "platform_variance", "certification_pressure", "streaming_pressure"]
normalized = platform_profiles.set_index("platform")[resource_cols].copy()
for col in resource_cols:
    values = normalized[col].astype(float)
    normalized[col] = (values - values.min()) / (values.max() - values.min())

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2), gridspec_kw={"width_ratios": [1.35, 1.0]})
ax = axes[0]
im = ax.imshow(normalized.values, cmap="YlGnBu", aspect="auto", vmin=0, vmax=1)
ax.set_xticks(range(len(resource_cols)))
ax.set_xticklabels([c.replace("_", "\n") for c in resource_cols], fontsize=8)
ax.set_yticks(range(len(normalized.index)))
ax.set_yticklabels(normalized.index, fontsize=9)
ax.set_title("Normalized platform pressure and capacity", fontsize=12)
for row in range(normalized.shape[0]):
    for col in range(normalized.shape[1]):
        ax.text(col, row, f"{normalized.iloc[row, col]:.2f}", ha="center", va="center", fontsize=7, color="#1f2933")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax = axes[1]
bar_colors = [PALETTE["teal"] if fps == 30 else PALETTE["red"] for fps in platform_profiles["target_fps"]]
ax.barh(platform_profiles["platform"], platform_profiles["time_budget_ms"], color=bar_colors)
ax.axvline(1000 / 60, color=PALETTE["red"], linestyle="--", label="60 fps budget")
ax.axvline(1000 / 30, color=PALETTE["teal"], linestyle="--", label="30 fps budget")
ax.set_xlabel("Milliseconds per frame")
ax.set_title("Frame time available before all other work")
ax.invert_yaxis()
ax.legend(loc="lower right", fontsize=8)
style_axis(ax, "Frame budgets by target frame rate", xlabel="ms")
fig.tight_layout()
platform_fig = save_matplotlib(fig, TOPIC, "platform-resource-envelope.png")
plt.close(fig)
platform_table = save_table_csv(platform_profiles.to_dict("records"), TOPIC, "platform-resource-envelope.csv")
platform_checks = {
    "min_time_budget_ms": float(platform_profiles["time_budget_ms"].min()),
    "max_time_budget_ms": float(platform_profiles["time_budget_ms"].max()),
    "fps_60_budget_is_half_30": bool(np.isclose(1000 / 60, (1000 / 30) / 2)),
    "platform_count": int(len(platform_profiles)),
    "all_positive_budgets": bool((platform_profiles[["time_budget_ms", "gpu_capacity", "memory_gb", "bandwidth_gbps"]] > 0).all().all()),
}
platform_check_path = save_json(platform_checks, TOPIC, "platform-resource-checks.json")
display_artifact(platform_fig, width=950)
platform_profiles


## Processing time, command buffers, and bottlenecks

The source chapter emphasizes that rendering commands do not usually execute at the instant the CPU submits them. The CPU fills a command buffer; the GPU drains it. Buffering protects the GPU from starvation, but it also means that input sampled for one frame may not appear on screen until a later display refresh. That tradeoff is central to fast games.

The synthetic model below separates CPU simulation/submission, GPU vertex work, GPU pixel work, texture fetch, bandwidth, and post processing. The important invariant is that frame time is controlled by the maximum of the active resource lanes, not the sum of every lane. If pixel shading is the bottleneck, removing a small amount of CPU work will not change the displayed frame rate. If the command buffer grows, the starvation risk drops but input latency rises.


In [ ]:
frame_budget_ms = {30: 1000 / 30, 60: 1000 / 60}
resource_pool_ms = {
    "CPU total": 5.2 + 3.8 + 2.6,
    "GPU programmable": 13.4 + 4.7 + 3.5,
    "Texture units": 5.6,
    "Memory bandwidth": 7.8,
}
baseline_frame_ms = max(resource_pool_ms.values())
non_bottleneck_reduced = dict(resource_pool_ms)
non_bottleneck_reduced["CPU total"] *= 0.7
non_bottleneck_frame_ms = max(non_bottleneck_reduced.values())
bottleneck_reduced = dict(resource_pool_ms)
bottleneck_reduced["GPU programmable"] *= 0.72
bottleneck_frame_ms = max(bottleneck_reduced.values())

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.0), gridspec_kw={"width_ratios": [1.15, 1.0]})
ax = axes[0]
rows = [("CPU builds frame n", 0, 13.0, PALETTE["blue"]), ("GPU renders frame n-1", 5.0, 21.6, PALETTE["teal"]), ("Display scans out n-2", 16.7, 16.7, PALETTE["gold"])]
for y, (label, start, width, color) in enumerate(rows):
    ax.broken_barh([(start, width)], (y - 0.32, 0.64), facecolors=color, alpha=0.9)
    ax.text(start + width / 2, y, label, ha="center", va="center", color="white", fontsize=9)
for x in [0, 16.7, 33.3, 50.0]:
    ax.axvline(x, color="#c7d0d9", linestyle="--", linewidth=0.9)
    ax.text(x, 2.55, f"{x:.1f} ms", ha="center", fontsize=8, color=PALETTE["ink"])
ax.annotate("one queued frame\nlow starvation, higher latency", xy=(20, 1), xytext=(28, 0.15), arrowprops={"arrowstyle": "->", "color": PALETTE["red"]}, fontsize=9, color=PALETTE["ink"])
ax.set_xlim(-1, 52)
ax.set_ylim(-0.8, 2.8)
ax.set_yticks([])
style_axis(ax, "Command buffer timeline", xlabel="time")

ax = axes[1]
labels = list(resource_pool_ms.keys())
values = [resource_pool_ms[k] for k in labels]
colors = [PALETTE["red"] if v == baseline_frame_ms else PALETTE["gray"] for v in values]
ax.barh(labels, values, color=colors)
ax.axvline(frame_budget_ms[60], color=PALETTE["red"], linestyle="--", label="60 fps")
ax.axvline(frame_budget_ms[30], color=PALETTE["teal"], linestyle="--", label="30 fps")
ax.set_xlabel("Synthetic milliseconds")
ax.set_title("Only the largest resource pool controls frame time")
ax.legend(fontsize=8)
for i, v in enumerate(values):
    ax.text(v + 0.2, i, f"{v:.1f}", va="center", fontsize=8)
style_axis(ax, "Bottleneck measurement", xlabel="ms")
fig.tight_layout()
frame_fig = save_matplotlib(fig, TOPIC, "cpu-gpu-command-buffer-latency.png")
plt.close(fig)

frame_checks = {
    "budget_30fps_ms": frame_budget_ms[30],
    "budget_60fps_ms": frame_budget_ms[60],
    "baseline_frame_ms": baseline_frame_ms,
    "baseline_hits_30fps": bool(baseline_frame_ms <= frame_budget_ms[30]),
    "baseline_hits_60fps": bool(baseline_frame_ms <= frame_budget_ms[60]),
    "non_bottleneck_reduction_changes_frame_ms": bool(not np.isclose(non_bottleneck_frame_ms, baseline_frame_ms)),
    "bottleneck_reduction_improves_frame_ms": bool(bottleneck_frame_ms < baseline_frame_ms),
    "bottleneck_resource": max(resource_pool_ms, key=resource_pool_ms.get),
}
frame_check_path = save_json(frame_checks, TOPIC, "frame-budget-command-buffer-checks.json")
display_artifact(frame_fig, width=950)
frame_checks


## Game type drives rendering choices

Game genre is not just marketing vocabulary. It predicts camera control, the amount of world visible per frame, how many moving foreground objects appear, how much lighting can be precomputed, and how much unique content must be resident or streamable. A fighting game can spend lavish detail on a small arena and two characters. A sandbox or persistent online world must stream, cull, and choose LOD aggressively because the player can move toward distant objects. A rhythm game or twitch shooter usually pays a strict latency tax by targeting 60 fps.

The interactive Plotly artifact is intentionally a decision map, not a ranking. Select a game type in the legend and compare it across axes. High precomputed lighting favors static backgrounds and fixed levels; high streaming pressure points toward world partitioning, LOD, and aggressive memory budgeting; many visible actors point toward instancing, animation LOD, and cheaper materials.


In [ ]:
game_types = pd.DataFrame([
    {"game_type": "Fighting arena", "target_fps": 60, "camera_freedom": 1, "visible_world_span": 1, "visible_actors": 2, "precomputed_lighting": 9, "streaming_pressure": 1, "unique_asset_pressure": 4, "stylization_room": 5},
    {"game_type": "First-person shooter", "target_fps": 60, "camera_freedom": 6, "visible_world_span": 5, "visible_actors": 12, "precomputed_lighting": 6, "streaming_pressure": 5, "unique_asset_pressure": 7, "stylization_room": 4},
    {"game_type": "Open-world sandbox", "target_fps": 30, "camera_freedom": 10, "visible_world_span": 10, "visible_actors": 40, "precomputed_lighting": 2, "streaming_pressure": 10, "unique_asset_pressure": 10, "stylization_room": 6},
    {"game_type": "Real-time strategy", "target_fps": 30, "camera_freedom": 8, "visible_world_span": 8, "visible_actors": 160, "precomputed_lighting": 5, "streaming_pressure": 6, "unique_asset_pressure": 7, "stylization_room": 7},
    {"game_type": "Puzzle or casual", "target_fps": 30, "camera_freedom": 2, "visible_world_span": 2, "visible_actors": 8, "precomputed_lighting": 8, "streaming_pressure": 2, "unique_asset_pressure": 3, "stylization_room": 9},
    {"game_type": "Two-and-a-half-D platformer", "target_fps": 30, "camera_freedom": 3, "visible_world_span": 4, "visible_actors": 14, "precomputed_lighting": 7, "streaming_pressure": 4, "unique_asset_pressure": 6, "stylization_room": 8},
    {"game_type": "Persistent online world", "target_fps": 30, "camera_freedom": 10, "visible_world_span": 9, "visible_actors": 100, "precomputed_lighting": 3, "streaming_pressure": 9, "unique_asset_pressure": 9, "stylization_room": 6},
])
game_types["frame_budget_ms"] = 1000 / game_types["target_fps"]
game_types["actor_storage_pressure"] = np.log2(game_types["visible_actors"] + 1) * game_types["unique_asset_pressure"] / 10

dimensions = ["target_fps", "camera_freedom", "visible_world_span", "visible_actors", "precomputed_lighting", "streaming_pressure", "unique_asset_pressure", "stylization_room"]
fig = px.parallel_coordinates(game_types, dimensions=dimensions, color="streaming_pressure", labels={col: col.replace("_", " ") for col in dimensions}, color_continuous_scale="Viridis", title="Rendering constraints implied by game type")
fig.update_layout(height=520, margin={"l": 80, "r": 50, "t": 70, "b": 35})
game_html = save_plotly_html(fig, TOPIC, "game-type-rendering-choice-map.html", include_plotlyjs=True)
game_table = save_table_csv(game_types.to_dict("records"), TOPIC, "game-type-rendering-model.csv")
game_checks = {
    "game_type_count": int(len(game_types)),
    "latency_sensitive_60fps": sorted(game_types.loc[game_types["target_fps"] == 60, "game_type"].tolist()),
    "open_world_has_highest_streaming_pressure": bool(game_types.sort_values("streaming_pressure").iloc[-1]["game_type"] in {"Open-world sandbox", "Persistent online world"}),
    "max_visible_actors": int(game_types["visible_actors"].max()),
    "min_frame_budget_ms": float(game_types["frame_budget_ms"].min()),
}
game_check_path = save_json(game_checks, TOPIC, "game-type-rendering-checks.json")
display_artifact(game_html, width="100%", height=560)
game_types


## Optimization: culling and LOD

The fastest object is the one not submitted to the graphics API. Frustum culling removes objects outside the camera volume; higher-level occlusion methods remove objects hidden by nearer geometry; LOD changes the representation of visible objects when their screen coverage is small. These techniques save different resources. Culling can save CPU submission, vertex work, pixel work, and bandwidth. LOD mostly saves vertex work and sometimes memory bandwidth, but it must preserve the silhouette and visible material behavior well enough that the switch is not distracting.

The experiment below creates a reproducible top-down scene. It computes a simple view-frustum predicate, a synthetic occlusion predicate, and a screen-coverage LOD rule. The left panel shows which objects survive visibility tests. The right panel shows how draw calls, triangle count, and pixel work change as the renderer applies each optimization step.


In [ ]:
rng = np.random.default_rng(22)
object_count = 520
positions = np.column_stack([rng.uniform(5, 115, object_count), rng.uniform(-75, 75, object_count)])
radii = rng.uniform(0.4, 2.2, object_count)
base_triangles = rng.integers(900, 6200, object_count)
distances = np.linalg.norm(positions, axis=1)
angles = np.degrees(np.arctan2(positions[:, 1], positions[:, 0]))
frustum_mask = (np.abs(angles) <= 34) & (distances <= 105)
occlusion_score = (distances / distances.max()) * np.exp(-(angles / 23) ** 2)
occlusion_mask = frustum_mask & (rng.random(object_count) < 0.34 * occlusion_score)
visible_mask = frustum_mask & ~occlusion_mask
screen_coverage = radii / np.maximum(distances, 1) * 100
lod_level = np.select([screen_coverage > 2.2, screen_coverage > 0.85], ["high", "medium"], default="low")
lod_factor = np.select([lod_level == "high", lod_level == "medium"], [1.0, 0.35], default=0.12)
lod_triangles = (base_triangles * lod_factor).astype(int)

stages = pd.DataFrame([
    {"stage": "All scene objects", "draw_calls": object_count, "triangles": int(base_triangles.sum()), "pixel_work": float((base_triangles * screen_coverage).sum())},
    {"stage": "After frustum cull", "draw_calls": int(frustum_mask.sum()), "triangles": int(base_triangles[frustum_mask].sum()), "pixel_work": float((base_triangles[frustum_mask] * screen_coverage[frustum_mask]).sum())},
    {"stage": "After occlusion cull", "draw_calls": int(visible_mask.sum()), "triangles": int(base_triangles[visible_mask].sum()), "pixel_work": float((base_triangles[visible_mask] * screen_coverage[visible_mask]).sum())},
    {"stage": "After LOD", "draw_calls": int(visible_mask.sum()), "triangles": int(lod_triangles[visible_mask].sum()), "pixel_work": float((lod_triangles[visible_mask] * screen_coverage[visible_mask]).sum())},
])

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.7), gridspec_kw={"width_ratios": [1.05, 1.0]})
ax = axes[0]
culled = ~frustum_mask
ax.scatter(positions[culled, 0], positions[culled, 1], s=9, color="#cbd5df", label="outside frustum")
ax.scatter(positions[occlusion_mask, 0], positions[occlusion_mask, 1], s=13, color=PALETTE["gold"], label="occluded")
visible_colors = np.where(lod_level[visible_mask] == "high", PALETTE["red"], np.where(lod_level[visible_mask] == "medium", PALETTE["blue"], PALETTE["teal"]))
ax.scatter(positions[visible_mask, 0], positions[visible_mask, 1], s=16, c=visible_colors, label="visible LOD")
for theta in [-34, 34]:
    ax.plot([0, 105 * math.cos(math.radians(theta))], [0, 105 * math.sin(math.radians(theta))], color=PALETTE["ink"], linewidth=1.2)
arc_theta = np.linspace(-34, 34, 80)
ax.plot(105 * np.cos(np.radians(arc_theta)), 105 * np.sin(np.radians(arc_theta)), color=PALETTE["ink"], linewidth=1.0)
ax.scatter([0], [0], marker="^", s=80, color=PALETTE["ink"], label="camera")
ax.set_xlim(-5, 120)
ax.set_ylim(-80, 80)
ax.legend(loc="upper right", fontsize=8)
style_axis(ax, "Synthetic visibility set", equal=True, xlabel="world x", ylabel="world y")

ax = axes[1]
plot_stages = stages.set_index("stage")[["draw_calls", "triangles", "pixel_work"]].copy()
plot_stages["triangles"] /= 1000
plot_stages["pixel_work"] /= 100000
plot_stages.plot(kind="bar", ax=ax, color=[PALETTE["gray"], PALETTE["blue"], PALETTE["red"]])
ax.set_ylabel("normalized units")
ax.set_xticklabels(plot_stages.index, rotation=20, ha="right")
style_axis(ax, "Budget saved by rejection and simpler meshes")
ax.legend(["draw calls", "triangles x1000", "pixel work x100k"], fontsize=8)
fig.tight_layout()
culling_fig = save_matplotlib(fig, TOPIC, "culling-lod-visibility-budget.png")
plt.close(fig)

coverage_by_lod = pd.DataFrame({"distance": distances, "coverage": screen_coverage, "lod": lod_level, "triangles": lod_triangles})
mean_triangles_by_lod = coverage_by_lod.groupby("lod")["triangles"].mean().to_dict()
culling_checks = {
    "object_count": int(object_count),
    "frustum_visible_count": int(frustum_mask.sum()),
    "occlusion_culled_count": int(occlusion_mask.sum()),
    "final_visible_count": int(visible_mask.sum()),
    "triangles_before": int(stages.iloc[0]["triangles"]),
    "triangles_after_lod": int(stages.iloc[-1]["triangles"]),
    "lod_reduces_triangles": bool(stages.iloc[-1]["triangles"] < stages.iloc[2]["triangles"]),
    "frustum_culling_reduces_draws": bool(stages.iloc[1]["draw_calls"] < stages.iloc[0]["draw_calls"]),
    "lod_mean_triangle_order_high_medium_low": bool(mean_triangles_by_lod["high"] > mean_triangles_by_lod["medium"] > mean_triangles_by_lod["low"]),
}
culling_check_path = save_json(culling_checks, TOPIC, "culling-lod-checks.json")
display_artifact(culling_fig, width=950)
stages


## Texture, mip, and normal-map budgets

Textures usually dominate game graphics memory. A single surface may need diffuse color, normal direction, specular response, smoothness, emissive masks, or other material channels. Mipmaps improve texture filtering and reduce aliasing, but they also add storage. For a large square texture, the full mip chain costs about one third more than the base level because the areas form a geometric series: `1 + 1/4 + 1/16 + ... = 4/3`.

Normal maps are a key game-graphics compromise. Fine detail is sculpted or otherwise produced at high resolution, then baked into a texture that perturbs shading normals on a cheaper mesh. The normal map does not add silhouette detail, and it spends texture memory and bandwidth, but it can save millions of triangles. The generated normal map below is synthetic: a height field is differentiated, normalized, encoded into RGB, and checked after decoding.


In [ ]:
def mip_chain(width: int, height: int, channels: int = 4, bytes_per_channel: int = 1) -> pd.DataFrame:
    rows = []
    level = 0
    w, h = width, height
    while True:
        rows.append({"level": level, "width": w, "height": h, "bytes": int(w * h * channels * bytes_per_channel)})
        if w == 1 and h == 1:
            break
        w = max(1, w // 2)
        h = max(1, h // 2)
        level += 1
    return pd.DataFrame(rows)

size = 256
grid = np.linspace(0, 2 * np.pi, size)
xx, yy = np.meshgrid(grid, grid)
height = 0.38 * np.sin(2.5 * xx) * np.cos(1.5 * yy) + 0.12 * np.sin(8 * xx + 2 * yy)
dy, dx = np.gradient(height)
normals = np.dstack([-4.0 * dx, -4.0 * dy, np.ones_like(height)])
normals /= np.linalg.norm(normals, axis=2, keepdims=True)
encoded = ((normals * 0.5 + 0.5) * 255).round().clip(0, 255).astype(np.uint8)
normal_map_path = save_image(encoded, TOPIC, "synthetic-baked-normal-map.png")
decoded = encoded.astype(float) / 255 * 2 - 1
decoded_lengths = np.linalg.norm(decoded, axis=2)

mip_df = mip_chain(2048, 2048, channels=4)
base_bytes = int(mip_df.iloc[0]["bytes"])
total_mip_bytes = int(mip_df["bytes"].sum())
texture_sets = pd.DataFrame([
    {"map": "diffuse RGBA", "size": "2048", "bytes": total_mip_bytes},
    {"map": "normal RGB/RGBA", "size": "2048", "bytes": total_mip_bytes},
    {"map": "specular", "size": "1024", "bytes": int(mip_chain(1024, 1024, channels=4)["bytes"].sum())},
    {"map": "smoothness", "size": "1024", "bytes": int(mip_chain(1024, 1024, channels=1)["bytes"].sum())},
    {"map": "emissive or mask", "size": "512", "bytes": int(mip_chain(512, 512, channels=1)["bytes"].sum())},
])
texture_sets["MiB"] = texture_sets["bytes"] / (1024 ** 2)

fig, axes = plt.subplots(1, 3, figsize=(14.2, 4.9), gridspec_kw={"width_ratios": [1.0, 1.0, 1.15]})
ax = axes[0]
ax.imshow(encoded)
ax.set_title("Encoded normal map")
ax.axis("off")
ax.text(0.02, 0.98, "RGB stores x,y,z normal components", transform=ax.transAxes, va="top", fontsize=8, bbox={"facecolor": "white", "edgecolor": "#d0d7de", "alpha": 0.9})
ax = axes[1]
ax.plot(mip_df["level"], mip_df["bytes"] / base_bytes, marker="o", color=PALETTE["blue"])
ax.fill_between(mip_df["level"], mip_df["bytes"] / base_bytes, color=PALETTE["blue"], alpha=0.16)
ax.set_yscale("log")
style_axis(ax, "Mip levels shrink by area", xlabel="mip level", ylabel="fraction of base bytes")
ax = axes[2]
ax.barh(texture_sets["map"], texture_sets["MiB"], color=[PALETTE["teal"], PALETTE["red"], PALETTE["blue"], PALETTE["gold"], PALETTE["gray"]])
ax.set_xlabel("MiB including mip chain")
ax.invert_yaxis()
style_axis(ax, "One hero material texture budget", xlabel="MiB")
for i, value in enumerate(texture_sets["MiB"]):
    ax.text(value + 0.15, i, f"{value:.1f}", va="center", fontsize=8)
fig.tight_layout()
texture_fig = save_matplotlib(fig, TOPIC, "texture-normal-map-memory-budget.png")
plt.close(fig)

texture_table = save_table_csv(texture_sets.to_dict("records"), TOPIC, "texture-normal-map-budget.csv")
texture_checks = {
    "base_2048_rgba_mib": base_bytes / (1024 ** 2),
    "mip_chain_ratio": total_mip_bytes / base_bytes,
    "mip_chain_below_four_thirds_plus_rounding": bool(total_mip_bytes / base_bytes <= 1.334),
    "decoded_normal_length_min": float(decoded_lengths.min()),
    "decoded_normal_length_max": float(decoded_lengths.max()),
    "decoded_normal_length_mean": float(decoded_lengths.mean()),
    "total_hero_material_mib": float(texture_sets["MiB"].sum()),
}
texture_check_path = save_json(texture_checks, TOPIC, "texture-normal-map-checks.json")
display_artifact(texture_fig, width=950)
display_artifact(normal_map_path, width=360)
texture_sets


## Asset pipeline and production workflow

A game graphics technique is not finished when the runtime shader compiles. The technique must fit an asset pipeline. Artists need tools to create inputs, preview results, revise work, and see failures early. A normal-map workflow, for example, involves modeling, UV parameterization, high-detail sculpting, baking, texture painting, shader binding, and engine import. Each stage can require approval and iteration. Character assets add rigging, skinning weights, animations, transitions, and sometimes morph targets.

The dependency graph below combines production phases with asset creation. It is a proof scaffold for workflow reasoning: if a node depends on another node, a late change upstream can invalidate downstream work. The graph check asserts acyclicity because a cyclic production graph is a schedule smell: no stage can complete because each waits on another.


In [ ]:
G = nx.DiGraph()
edges = [
    ("game concept", "prototype"), ("prototype", "greenlight"), ("greenlight", "pre-production"), ("pre-production", "engine and tools ready"),
    ("engine and tools ready", "full production"), ("full production", "alpha"), ("alpha", "beta"), ("beta", "gold release"),
    ("gold release", "patch or DLC"), ("pre-production", "next project seed"),
    ("initial mesh model", "UV parameterization"), ("initial mesh model", "animation-ready topology"), ("UV parameterization", "detail sculpt"),
    ("detail sculpt", "normal map bake"), ("UV parameterization", "paint material maps"), ("normal map bake", "shader assignment"),
    ("paint material maps", "shader assignment"), ("shader assignment", "engine import"), ("engine import", "lighting bake"),
    ("animation-ready topology", "rigging"), ("rigging", "skinning weights"), ("skinning weights", "animation clips"),
    ("animation clips", "engine import"), ("engine import", "profiled build"), ("lighting bake", "profiled build"),
    ("profiled build", "full production"), ("profiled build", "alpha"),
]
G.add_edges_from(edges)
assert nx.is_directed_acyclic_graph(G)
layers = {
    "game concept": 0, "prototype": 1, "greenlight": 2, "pre-production": 3, "engine and tools ready": 4, "full production": 5, "alpha": 6, "beta": 7, "gold release": 8, "patch or DLC": 9, "next project seed": 5,
    "initial mesh model": 0, "UV parameterization": 1, "animation-ready topology": 1, "detail sculpt": 2, "normal map bake": 3, "paint material maps": 3, "shader assignment": 4, "engine import": 5, "lighting bake": 6, "rigging": 2, "skinning weights": 3, "animation clips": 4, "profiled build": 6,
}
production_nodes = {"game concept", "prototype", "greenlight", "pre-production", "engine and tools ready", "full production", "alpha", "beta", "gold release", "patch or DLC", "next project seed"}
pos = {node: (layers[node], 1.0 if node in production_nodes else 0.0) for node in G.nodes}
pos["profiled build"] = (6, 0.2)
pos["engine import"] = (5, 0.2)

fig, ax = plt.subplots(figsize=(15, 6.2))
node_colors = [PALETTE["red"] if node in {"profiled build", "engine import", "engine and tools ready"} else (PALETTE["blue"] if node in production_nodes else PALETTE["teal"]) for node in G.nodes]
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=12, edge_color="#8391a1", width=1.1, connectionstyle="arc3,rad=0.03")
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=1350, edgecolors="white", linewidths=1.2)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=7, font_color="white")
ax.text(0, 1.35, "production workflow", fontsize=11, color=PALETTE["ink"], weight="bold")
ax.text(0, -0.35, "asset creation workflow", fontsize=11, color=PALETTE["ink"], weight="bold")
ax.set_axis_off()
ax.set_title("Graphics choices become production dependencies", fontsize=13)
fig.tight_layout()
pipeline_fig = save_matplotlib(fig, TOPIC, "asset-pipeline-production-dag.png")
plt.close(fig)

pipeline_checks = {
    "node_count": int(G.number_of_nodes()),
    "edge_count": int(G.number_of_edges()),
    "is_dag": bool(nx.is_directed_acyclic_graph(G)),
    "critical_path_length_edges": int(nx.dag_longest_path_length(G)),
    "critical_path": nx.dag_longest_path(G),
    "engine_import_precedes_profiled_build": bool(nx.has_path(G, "engine import", "profiled build")),
    "normal_map_bake_precedes_shader_assignment": bool(nx.has_path(G, "normal map bake", "shader assignment")),
}
pipeline_check_path = save_json(pipeline_checks, TOPIC, "asset-pipeline-checks.json")
display_artifact(pipeline_fig, width=1000)
pipeline_checks


## Applied lab: profile before choosing quality knobs

A common mistake is to lower a visible quality setting without knowing whether it targets the bottleneck. The lab below builds a small synthetic profiler. It sweeps resolution scale, antialiasing samples, shadow quality, texture tier, LOD bias, and culling mode. Each setting receives an estimated CPU cost, GPU shader cost, bandwidth cost, memory cost, latency cost, and visual quality score. The selected setting is the highest quality configuration that fits a target platform.

This is not a replacement for measuring a real game. It is a compact model of the decision process described in the chapter: profile the frame, identify the resource pool that misses the budget, and choose the smallest quality concession that fixes the bottleneck. The generated table can be filtered or modified as a follow-up exercise.


In [ ]:
resolution_scales = [0.70, 0.85, 1.00]
aa_samples = [1, 2, 4]
shadow_quality = ["off", "medium", "high"]
texture_tiers = ["low", "medium", "high"]
lod_biases = [0.8, 1.0, 1.25]
culling_modes = ["frustum", "frustum+occlusion"]
shadow_gpu = {"off": 0.0, "medium": 2.8, "high": 5.6}
shadow_cpu = {"off": 0.0, "medium": 0.7, "high": 1.2}
shadow_quality_score = {"off": 0, "medium": 8, "high": 14}
texture_memory = {"low": 650, "medium": 1200, "high": 2400}
texture_bandwidth = {"low": 2.0, "medium": 3.4, "high": 5.9}
texture_score = {"low": 4, "medium": 9, "high": 14}
culling_cpu = {"frustum": 2.2, "frustum+occlusion": 3.3}
culling_gpu_multiplier = {"frustum": 1.0, "frustum+occlusion": 0.74}
culling_score = {"frustum": 0, "frustum+occlusion": 3}

rows = []
for res in resolution_scales:
    for aa in aa_samples:
        for shadow in shadow_quality:
            for texture in texture_tiers:
                for lod in lod_biases:
                    for culling in culling_modes:
                        pixel_scale = res ** 2 * (1 + 0.16 * (aa - 1))
                        cpu_ms = 4.6 + shadow_cpu[shadow] + culling_cpu[culling] + 0.6 / lod
                        gpu_ms = (7.5 * pixel_scale + shadow_gpu[shadow] + 5.0 / lod) * culling_gpu_multiplier[culling]
                        bandwidth_ms = (2.2 * pixel_scale + texture_bandwidth[texture]) * culling_gpu_multiplier[culling]
                        memory_mb = texture_memory[texture] + 220 / lod + (130 if shadow != "off" else 0)
                        latency_frames = 1 + (1 if aa == 4 and shadow == "high" else 0)
                        quality = 35 * res + 3 * math.log2(aa) + shadow_quality_score[shadow] + texture_score[texture] + 8 / lod + culling_score[culling]
                        rows.append({"resolution_scale": res, "aa_samples": aa, "shadow_quality": shadow, "texture_tier": texture, "lod_bias": lod, "culling_mode": culling, "cpu_ms": cpu_ms, "gpu_ms": gpu_ms, "bandwidth_ms": bandwidth_ms, "frame_ms": max(cpu_ms, gpu_ms, bandwidth_ms), "memory_mb": memory_mb, "latency_frames": latency_frames, "quality_score": quality})
settings = pd.DataFrame(rows)

def choose_setting(target_fps: int, memory_limit_mb: int, max_latency_frames: int = 2) -> pd.Series:
    budget = 1000 / target_fps
    feasible = settings[(settings["frame_ms"] <= budget) & (settings["memory_mb"] <= memory_limit_mb) & (settings["latency_frames"] <= max_latency_frames)]
    if feasible.empty:
        raise ValueError("No setting fits target")
    return feasible.sort_values(["quality_score", "frame_ms"], ascending=[False, True]).iloc[0]

chosen_30 = choose_setting(30, 1800, max_latency_frames=2)
chosen_60 = choose_setting(60, 1200, max_latency_frames=1)
quality_table = pd.DataFrame([chosen_30, chosen_60], index=["30 fps memory 1800", "60 fps memory 1200"]).reset_index(names="target")
quality_table_path = save_table_csv(quality_table.to_dict("records"), TOPIC, "quality-knob-fit-results.csv")

fig, axes = plt.subplots(1, 2, figsize=(13.6, 5.2), gridspec_kw={"width_ratios": [1.0, 1.08]})
ax = axes[0]
bar_data = quality_table.set_index("target")[["cpu_ms", "gpu_ms", "bandwidth_ms"]]
bar_data.plot(kind="bar", ax=ax, color=[PALETTE["blue"], PALETTE["red"], PALETTE["gold"]])
ax.axhline(1000 / 30, color=PALETTE["teal"], linestyle="--", linewidth=1, label="30 fps budget")
ax.axhline(1000 / 60, color=PALETTE["red"], linestyle=":", linewidth=1, label="60 fps budget")
ax.set_ylabel("milliseconds")
ax.set_xticklabels(bar_data.index, rotation=0)
style_axis(ax, "Chosen settings fit their resource pools")
ax.legend(fontsize=8)

ax = axes[1]
subset = settings.sample(450, random_state=22)
scatter = ax.scatter(subset["frame_ms"], subset["quality_score"], c=subset["memory_mb"], s=18, cmap="viridis", alpha=0.7)
ax.scatter([chosen_30["frame_ms"], chosen_60["frame_ms"]], [chosen_30["quality_score"], chosen_60["quality_score"]], marker="*", s=240, color=PALETTE["red"], edgecolors="white", linewidths=1.0, label="selected")
ax.axvline(1000 / 30, color=PALETTE["teal"], linestyle="--", linewidth=1)
ax.axvline(1000 / 60, color=PALETTE["red"], linestyle=":", linewidth=1)
ax.set_xlabel("frame ms")
ax.set_ylabel("synthetic quality score")
ax.legend(fontsize=8)
style_axis(ax, "Quality versus measured frame cost")
fig.colorbar(scatter, ax=ax, label="memory MB")
fig.tight_layout()
profiling_fig = save_matplotlib(fig, TOPIC, "profiling-optimization-lab.png")
plt.close(fig)

top_quality = settings.sort_values("quality_score").iloc[-1]
profiling_checks = {
    "candidate_settings": int(len(settings)),
    "chosen_30fps_frame_ms": float(chosen_30["frame_ms"]),
    "chosen_60fps_frame_ms": float(chosen_60["frame_ms"]),
    "chosen_30fps_fits_budget": bool(chosen_30["frame_ms"] <= 1000 / 30 and chosen_30["memory_mb"] <= 1800),
    "chosen_60fps_fits_budget": bool(chosen_60["frame_ms"] <= 1000 / 60 and chosen_60["memory_mb"] <= 1200 and chosen_60["latency_frames"] <= 1),
    "highest_quality_candidate_exceeds_60fps_or_memory": bool(top_quality["frame_ms"] > 1000 / 60 or top_quality["memory_mb"] > 1200),
}
profiling_check_path = save_json(profiling_checks, TOPIC, "profiling-lab-checks.json")
display_artifact(profiling_fig, width=950)
quality_table


## Sanity checks

The final cell collects the chapter checks. It asserts the identities that are easy to forget when tuning a renderer: the 60 fps budget is half of the 30 fps budget; frame time follows the bottleneck resource; culling and LOD reduce submitted work; mipmaps follow the geometric memory rule; decoded normal vectors remain close to unit length; the production graph is acyclic; and the profiler selects settings that fit their target budgets.

It also checks that every generated artifact exists and has nonzero size. These checks are intentionally local to Chapter 22 so they can run under `nbclient` without touching other chapters.


In [ ]:
artifact_paths = [
    storyboard_path, source_span_path, platform_fig, platform_table, platform_check_path, frame_fig, frame_check_path,
    game_html, game_table, game_check_path, culling_fig, culling_check_path, texture_fig, normal_map_path, texture_table,
    texture_check_path, pipeline_fig, pipeline_check_path, profiling_fig, quality_table_path, profiling_check_path,
]

assert platform_checks["fps_60_budget_is_half_30"]
assert platform_checks["all_positive_budgets"]
assert frame_checks["baseline_hits_30fps"] and not frame_checks["baseline_hits_60fps"]
assert frame_checks["non_bottleneck_reduction_changes_frame_ms"] is False
assert frame_checks["bottleneck_reduction_improves_frame_ms"]
assert game_checks["game_type_count"] >= 6
assert culling_checks["lod_reduces_triangles"] and culling_checks["frustum_culling_reduces_draws"]
assert culling_checks["lod_mean_triangle_order_high_medium_low"]
assert texture_checks["mip_chain_below_four_thirds_plus_rounding"]
assert 0.98 <= texture_checks["decoded_normal_length_mean"] <= 1.02
assert pipeline_checks["is_dag"]
assert profiling_checks["chosen_30fps_fits_budget"] and profiling_checks["chosen_60fps_fits_budget"]

artifact_records = assert_artifacts(artifact_paths, min_bytes=1200)
image_records = [image_stats(path) for path in [platform_fig, frame_fig, culling_fig, texture_fig, normal_map_path, pipeline_fig, profiling_fig]]
for record in image_records:
    assert record["max_channel_stddev"] > 1.0, f"blank-looking image: {record['path']}"

final_sanity = {
    "source_span": SOURCE_SPAN,
    "artifact_count": len(artifact_records),
    "image_count": len(image_records),
    "nonblank_images": sum(record["max_channel_stddev"] > 1.0 for record in image_records),
    "core_checks": {
        "platform": platform_checks,
        "frame_budget": frame_checks,
        "game_type": game_checks,
        "culling_lod": culling_checks,
        "texture_normal_map": texture_checks,
        "asset_pipeline": pipeline_checks,
        "profiling_lab": profiling_checks,
    },
    "artifacts": artifact_records,
    "image_stats": image_records,
}
final_sanity_path = save_json(final_sanity, TOPIC, "final-sanity.json")
assert_artifacts([final_sanity_path], min_bytes=1200)
final_sanity_path.relative_to(BOOK_ROOT).as_posix()


## Takeaways

- Platform decisions are budget decisions. A fixed console, a variable PC, a virtual-machine platform, and a handheld platform ask the renderer to spend different mixtures of time, bandwidth, memory, abstraction, and engineering effort.
- Frame rate turns graphics into arithmetic. A 60 fps target gives about 16.7 ms for the whole frame, so buffering, culling, shader cost, and latency must be reasoned about together.
- Optimization starts with the bottleneck. Reducing a non-bottleneck resource may make the code cleaner, but it will not improve frame time until the limiting resource moves.
- Game type changes the rendering problem. Camera freedom, world size, actor count, precomputed lighting, and audience expectations all affect which techniques are worth their cost.
- LOD, culling, mipmaps, normal maps, and precomputed lighting are production choices as much as runtime tricks. They require authoring tools, preview paths, validation, and iteration budgets.
- A useful profiling lab records both visual quality and resource use. The best setting is the highest-quality setting that fits the target frame time, memory budget, and latency requirement.
